# 📊 INF01090 - Ciência de Dados - Model Evaluation Metrics

This notebook is a hands-on guide to understanding classification and regression metrics through real-world scenarios.


## 📂 Datasets Explored
In this lab, we use several standard datasets to ground our metrics in reality:

1.  **Handwritten Digits (`load_digits`)**: 1,797 images of numbers (0-9). Excellent for testing multi-class accuracy.
2.  **Breast Cancer Diagnostic (`load_breast_cancer`)**: Medical data used to predict if a tumor is malignant. Critical for studying Recall and Precision.
3.  **California Housing (`fetch_california_housing`)**: Real estate data regarding 20,640 neighborhoods. Used to predict property values.
4.  **Synthetic Data (`make_classification`/`make_regression`)**: Computer-generated data used to simulate specific scenarios, like extreme fraud imbalance (where 99% of transactions are legitimate).


## 🛠 Library Reference

*   **`sklearn`**: Our main library for machine learning models and metrics calculation.
*   **`altair`**: A declarative statistical visualization library. We use it to create interactive and modern charts.
*   **`train_test_split`**: Splits your data into a **training set** (to teach the model) and a **test set** (to see if it actually learned).


In [1]:
import numpy as np
import pandas as pd
import altair as alt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn import metrics
from sklearn.datasets import load_digits, load_breast_cancer, fetch_california_housing, make_classification, make_regression

# Setting seed for reproducibility
np.random.seed(42)

# Enable Altair rendering
alt.renderers.enable('default')

# Helper for Labeled Confusion Matrix
def plot_labeled_cm(y_true, y_pred, title):
    cm = metrics.confusion_matrix(y_true, y_pred)
    df = pd.DataFrame(cm).stack().reset_index()
    df.columns = ['true', 'predicted', 'count']

    # Label mapping (0: Malignant, 1: Benign)
    # Predicting 1 as positive:
    labels = {
        (1, 1): 'TP (Correct Benign)',
        (0, 0): 'TN (Correct Malignant)',
        (0, 1): 'FP (Missed Malignant!)',
        (1, 0): 'FN (False Alarm)'
    }
    df['type'] = df.apply(lambda x: labels.get((int(x['true']), int(x['predicted'])), ''), axis=1)

    base = alt.Chart(df).mark_rect().encode(
        x=alt.X('predicted:O', title='Predicted'),
        y=alt.Y('true:O', title='True'),
        color=alt.Color('count:Q', scale=alt.Scale(scheme='blues'), legend=None)
    ).properties(width=200, height=200, title=title)

    # Visibility threshold for text color
    max_count = int(df['count'].max())
    text_color = alt.condition(f'datum.count > {max_count/2}', alt.value('white'), alt.value('black'))

    text_count = base.mark_text(dy=-10, fontWeight='bold').encode(
        text='count:Q',
        color=text_color
    )
    text_type = base.mark_text(dy=10, fontSize=8).encode(
        text='type:N',
        color=text_color
    )

    return base + text_count + text_type

def plot_unlabeled_cm(y_true, y_pred, title):
    cm = metrics.confusion_matrix(y_true, y_pred)
    df = pd.DataFrame(cm).stack().reset_index()
    df.columns = ['true', 'predicted', 'count']

    base = alt.Chart(df).mark_rect().encode(
        x=alt.X('predicted:O', title='Predicted'),
        y=alt.Y('true:O', title='True'),
        color=alt.Color('count:Q', scale=alt.Scale(scheme='blues'), legend=None)
    ).properties(width=200, height=200, title=title)

    # Visibility threshold for text color
    max_count = int(df['count'].max())
    text_color = alt.condition(f'datum.count > {max_count/2}', alt.value('white'), alt.value('black'))

    text_count = base.mark_text(dy=-10, fontWeight='bold').encode(
        text='count:Q',
        color=text_color
    )

    return base + text_count


## 🔹 Part 1: Classification Metrics


### Case 1: Handwriting Recognition (USPS Zip Code Sorting)

#### 📮 The Business Context
In the 1990s, the **US Postal Service (USPS)** faced a massive challenge: sorting millions of handwritten letters by zip code. Manually reading every envelope was slow and expensive. They developed automated systems to "see" digits.

**Our Goal**: Build a model that can recognize handwritten numbers (0-9) with high enough reliability that we can trust it to route mail across the country.

#### 📏 The Metric Toolbox
1.  **Accuracy (Global Hit Rate)**:
    -   The percentage of images correctly identified.
    -   *Equation*: Perfect Guesses / Total Images.
    -   *Intuition*: "Out of 100 letters, how many went to the right city?"
2.  **Confusion Matrix (The Heatmap of Mistakes)**:
    -   A grid showing which numbers are being "confused."
    -   *Intuition*: Does the model think a **'7'** is actually a **'1'**? Does it struggle with **'3'** vs **'8'**? Identifying specific pairs of confusion is key to improving the sensors.
3.  **Top-k Accuracy (System Robustness)**:
    -   Often, a model provides a list of candidate probabilities (e.g., "I'm 60% sure it's a 7, but 30% sure it's a 1").
    -   **Top-5 Accuracy** means the answer is correct if the true digit is in the model's top 5 most likely guesses.
    -   *Real-world Use*: If the machine is unsure, it can flag the envelope for a human operator, showing them the top 5 guesses to speed up the manual override.


In [2]:
digits = load_digits()
digits_df = pd.DataFrame(digits.data, columns=digits.feature_names)
digits_df

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_6_6,pixel_6_7,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,5.0,0.0,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,9.0,0.0,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1792,0.0,0.0,4.0,10.0,13.0,6.0,0.0,0.0,0.0,1.0,...,4.0,0.0,0.0,0.0,2.0,14.0,15.0,9.0,0.0,0.0
1793,0.0,0.0,6.0,16.0,13.0,11.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,6.0,16.0,14.0,6.0,0.0,0.0
1794,0.0,0.0,1.0,11.0,15.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,9.0,13.0,6.0,0.0,0.0
1795,0.0,0.0,2.0,10.0,7.0,0.0,0.0,0.0,0.0,0.0,...,2.0,0.0,0.0,0.0,5.0,12.0,16.0,12.0,0.0,0.0


In [3]:
print(digits.DESCR)


.. _digits_dataset:

Optical recognition of handwritten digits dataset
--------------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 1797
:Number of Attributes: 64
:Attribute Information: 8x8 image of integer pixels in the range 0..16.
:Missing Attribute Values: None
:Creator: E. Alpaydin (alpaydin '@' boun.edu.tr)
:Date: July; 1998

This is a copy of the test set of the UCI ML hand-written digits datasets
https://archive.ics.uci.edu/ml/datasets/Optical+Recognition+of+Handwritten+Digits

The data set contains images of hand-written digits: 10 classes where
each class refers to a digit.

Preprocessing programs made available by NIST were used to extract
normalized bitmaps of handwritten digits from a preprinted form. From a
total of 43 people, 30 contributed to the training set and different 13
to the test set. 32x32 bitmaps are divided into nonoverlapping blocks of
4x4 and the number of on pixels are counted in each block. This generates
an in

In [4]:
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = LogisticRegression(max_iter=10000).fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)

print(f"Accuracy: {metrics.accuracy_score(y_test, y_pred):.4f}")

# Altair Confusion Matrix
cm = metrics.confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm).stack().reset_index()
cm_df.columns = ['true_label', 'predicted_label', 'count']

base = alt.Chart(cm_df).mark_rect().encode(
    x=alt.X('predicted_label:O', title='Predicted Label'),
    y=alt.Y('true_label:O', title='True Label'),
    color=alt.Color('count:Q', scale=alt.Scale(scheme='blues'), legend=None)
).properties(width=400, height=400, title='Confusion Matrix: Digit Recognition')

threshold = int(cm.max()) / 2
text = base.mark_text(baseline='middle').encode(
    text='count:Q',
    color=alt.condition(f'datum.count > {threshold}', alt.value('white'), alt.value('black'))
)
base + text

Accuracy: 0.9685


alt.LayerChart(...)

#### 🛠 Student Task 1
Compare `Top-2` and `Top-5` accuracy. Why does the score increase as `k` increases? Calculate Top-5 in the next cell.


In [5]:
top5 = metrics.top_k_accuracy_score(y_test, y_probs, k=5)
print(f"Top-5 Accuracy: {top5:.4f}")

Top-5 Accuracy: 1.0000


ANSWER TASK 1:  O Top-k considera um acerto se a resposta correta estiver entre as k principais previsões do modelo. Aumentar o k (por exemplo, de 2 para 5) dá ao modelo mais "chances" de incluir a resposta certa.

### Case 2: Medical Diagnosis (Mayo Clinic)
**Metric:** Precision, Recall, Specificity, F1.


#### 🧠 Concept: Probabilities vs. Labels

Machine learning models often calculate a **probability** (0 to 1) before picking a category.

*   **High Threshold (e.g., 0.8)**: The model is 'exclusive'. It only predicts a positive case when it is extremely sure. This **increases Precision** but often **decreases Recall** (misses many cases).
*   **Low Threshold (e.g., 0.1)**: The model is 'cautious'. It flags cases even with low probability to avoid missing any. This **increases Recall** but **decreases Precision** (more false alarms).

A classifier in sklearn uses functions to predict the final class for a given set of input data, such as:
*   **`clf.predict()`**: Automatically chooses the most likely category (usually checks if probability > 50%).
*   **`clf.predict_proba()`**: Shows you the raw percentages for each category.

In medicine, we might want to flag a patient even if there is only a 30% chance of a tumor (high caution). This is called 'adjusting the threshold'.

In [6]:
data = load_breast_cancer()
data_df = pd.DataFrame(data.data, columns=data.feature_names)
data_df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [7]:
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = LogisticRegression(solver='liblinear', C=0.01).fit(X_train, y_train)
y_probs = clf.predict_proba(X_test)[:, 1] # Probability of 'Benign'

#### Visual & Manual Trade-off Challenge

**Equations to use:**
-   **Precision** = $TP / (TP + FP)$
-   **Recall** = $TP / (TP + FN)$
-   **F1** = $2 \times (Precision \times Recall) / (Precision + Recall)$

**Challenge:**
1.  Run the code below and look at the **labeled confusion matrix** for the 0.3 and 0.7 thresholds.
2.  Identify the numbers for TP, FP, and FN.
3.  Observe the number of many Malignant cases we missed at different threshholds.
4.  **Calculate Precision and Recall manually**.
5.  Compare your manual results against the `sklearn` calculation in the next cell.


In [8]:
# 1. Lower threshold (0.3)
y_pred_low = (y_probs > 0.3).astype(int)
chart_low = plot_labeled_cm(y_test, y_pred_low, title='Threshold: 0.3')

# 2. Higher threshold (0.7)
y_pred_high = (y_probs > 0.7).astype(int)
chart_high = plot_labeled_cm(y_test, y_pred_high, title='Threshold: 0.7')

chart_low | chart_high


alt.HConcatChart(...)

#### 🛠 Student Task 2.1
**Verify your manual calculation for the 0.3 and 0.7 Thresholds below:**

In [13]:
# ANSWER TASK 2.1
# Calculate metrics using formula for the Low Threshold (0.3)
precision_low_manual = 0.9230769230769231
recall_low_manual = 1
f1_low_manual = 2*(precision_low_manual * recall_low_manual) / (precision_low_manual + recall_low_manual)
print(f"Formula Precision (0.3): {precision_low_manual :.4f}")
print(f"Formula Recall (0.3): {recall_low_manual:.4f}")
print(f"Formula F1-Score (0.3): {f1_low_manual:.4f}")
# Calculate metrics using formula for the High Threshold (0.7)
precision_high_manual = 0.9901960784313725
recall_high_manual = 0.9351851851851852
f1_high_manual = 2*(precision_high_manual * recall_high_manual) / (precision_high_manual + recall_high_manual)
print(f"Formula Precision (0.7): {precision_high_manual:.4f}")
print(f"Formula Recall (0.7): {recall_high_manual:.4f}")
print(f"Formula F1-Score (0.7): {f1_high_manual:.4f}")

Formula Precision (0.3): 0.9231
Formula Recall (0.3): 1.0000
Formula F1-Score (0.3): 0.9600
Formula Precision (0.7): 0.9902
Formula Recall (0.7): 0.9352
Formula F1-Score (0.7): 0.9619


In [14]:
# Calculate metrics using sklearn for the Low Threshold (0.3)
print(f"Sklearn Precision (0.3): {metrics.precision_score(y_test, y_pred_low):.4f}")
print(f"Sklearn Recall (0.3): {metrics.recall_score(y_test, y_pred_low):.4f}")
print(f"Sklearn F1-Score (0.3): {metrics.f1_score(y_test, y_pred_low):.4f}")
# Calculated metrics using sklearn for the High Threshold (0.7)
print(f"Sklearn Precision (0.7): {metrics.precision_score(y_test, y_pred_high):.4f}")
print(f"Sklearn Recall (0.7): {metrics.recall_score(y_test, y_pred_high):.4f}")
print(f"Sklearn F1-Score (0.7): {metrics.f1_score(y_test, y_pred_high):.4f}")


Sklearn Precision (0.3): 0.8852
Sklearn Recall (0.3): 1.0000
Sklearn F1-Score (0.3): 0.9391
Sklearn Precision (0.7): 1.0000
Sklearn Recall (0.7): 0.7407
Sklearn F1-Score (0.7): 0.8511


Now we calculate the metrics at two different thresholds and use the `plot_labeled_cm` function to see what happened to the 'Missed Malignant' cases (FP).

1.  **Lower the threshold to 0.1**: `y_pred_low = (y_probs > 0.1).astype(int)`
2.  **Raise the threshold to 0.9**: `y_pred_high = (y_probs > 0.9).astype(int)`

Compare the two charts below. How many Malignant cases did we miss at the 0.9 threshold?

In [15]:
# LOWER THRESHOLD (0.1)
y_pred_low = (y_probs > 0.1).astype(int)
chart_low = plot_labeled_cm(y_test, y_pred_low, title='Threshold: 0.1')

# HIGHER THRESHOLD (0.9)
y_pred_high = (y_probs > 0.9).astype(int)
chart_high = plot_labeled_cm(y_test, y_pred_high, title='Threshold: 0.9')

chart_low | chart_high

alt.HConcatChart(...)

**Calculate the metrics for the new thresholds**

In [16]:
# Calculate metrics using sklearn for the Low Threshold (0.1)
print(f"Sklearn Precision (0.1): {metrics.precision_score(y_test, y_pred_low):.4f}")
print(f"Sklearn Recall (0.1): {metrics.recall_score(y_test, y_pred_low):.4f}")
print(f"Sklearn F1-Score (0.1): {metrics.f1_score(y_test, y_pred_low):.4f}")
# Calculated metrics using sklearn for the High Threshold (0.9)
print(f"Sklearn Precision (0.9): {metrics.precision_score(y_test, y_pred_high):.4f}")
print(f"Sklearn Recall (0.9): {metrics.recall_score(y_test, y_pred_high):.4f}")
print(f"Sklearn F1-Score (0.9): {metrics.f1_score(y_test, y_pred_high):.4f}")

Sklearn Precision (0.1): 0.8852
Sklearn Recall (0.1): 1.0000
Sklearn F1-Score (0.1): 0.9391
Sklearn Precision (0.9): 1.0000
Sklearn Recall (0.9): 0.7407
Sklearn F1-Score (0.9): 0.8511


#### 🛠 Student Task 2.2
**Compare the precision, recall and F1 metrics. What do you notice about the trade-offs between the two thresholds? Which threshold would you choose for a real-world application, and why?**

ANSWER TASK 2.2: Há uma relação inversa entre Precisão e Recall. Com limiar baixo (0,1), o Recall é perfeito (1,0000), mas a Precisão cai (0,8852) devido a mais falsos positivos. Com limiar alto (0,9), a Precisão é perfeita (1,0000), porém o Recall diminui (0,7407), indicando mais falsos negativos. O F1-Score também cai em 0,9, pois a perda de casos supera o ganho em precisão.

### Case 3: Fraud Detection (PayPal)
**Metric:** ROC-AUC.


#### 🧬 Simulating Real-World Fraud
In fraud detection, victims are rare. We use `make_classification`, a `sklearn` function that generate a random n-class classification problem. We use `weights=[0.98, 0.02]` to create a dataset where **98%** of transactions are legitimate (`0`) and only **2%** are fraudulent (`1`).

To understand the 'Accuracy Trap', we compare our models against a **Baseline** that simply predicts 'No Fraud' for everyone.

#### 🏁 Models to Compare
1.  **Model A (Logistic Regression)**: A simpler, linear model.
2.  **Model B (Random Forest)**: A more complex model that combines many decision trees (a committee).
3.  **Model C (Dummy Baseline)**: A 'no-skill' model that guesses based on class frequency.

#### 📈 The ROC Curve: How it's Made
The **ROC (Receiver Operating Characteristic)** curve summarizes the model's performance across **EVERY possible threshold** (from 0 to 1).
1.  For each threshold, it calculates the **True Positive Rate** (Recall or Sensitivity) = $TP / (TP + FN)$ and the **False Positive Rate** (Specificity) = $FP / (FP + TN)$
2.  It plots these points on a graph.
3.  An **AUC (Area Under the Curve)** of **1.0** is a perfect model. An AUC of **0.5** (the diagonal line) is no better than guessing.

In [18]:
# 1. Setup Data
X, y = make_classification(n_samples=1000, n_classes=2, weights=[0.98, 0.02], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Train Model A (Logistic)
model_a = LogisticRegression().fit(X_train, y_train)
probs_a = model_a.predict_proba(X_test)[:, 1]
fpr_a, tpr_a, _ = metrics.roc_curve(y_test, probs_a)
auc_a = metrics.auc(fpr_a, tpr_a)

# 3. Train Model B (Random Forest)
model_b = RandomForestClassifier(random_state=42).fit(X_train, y_train)
probs_b = model_b.predict_proba(X_test)[:, 1]
fpr_b, tpr_b, _ = metrics.roc_curve(y_test, probs_b)
auc_b = metrics.auc(fpr_b, tpr_b)

# 4. Train Model C (Dummy Baseline)
model_c = DummyClassifier(strategy='stratified', random_state=42).fit(X_train, y_train)
probs_c = model_c.predict_proba(X_test)[:, 1]
fpr_c, tpr_c, _ = metrics.roc_curve(y_test, probs_c)
auc_c = metrics.auc(fpr_c, tpr_c)

# 5. Visualization
res_a = pd.DataFrame({'fpr': fpr_a, 'tpr': tpr_a, 'model': f'Logistic Reg (AUC={auc_a:.2f})'})
res_b = pd.DataFrame({'fpr': fpr_b, 'tpr': tpr_b, 'model': f'Random Forest (AUC={auc_b:.2f})'})
res_c = pd.DataFrame({'fpr': fpr_c, 'tpr': tpr_c, 'model': f'Dummy Baseline (AUC={auc_c:.2f})'})
plot_df = pd.concat([res_a, res_b, res_c])

line = alt.Chart(plot_df).mark_line().encode(
    x=alt.X('fpr', title='False Positive Rate'),
    y=alt.Y('tpr', title='True Positive Rate'),
    color='model:N'
)
diag = alt.Chart(pd.DataFrame({'x':[0,1], 'y':[0,1]})).mark_line(strokeDash=[5,5], color='black').encode(x='x', y='y')
(line + diag).properties(width=400, height=400, title='Comparative ROC Curve').interactive()

alt.LayerChart(...)

#### 🛠 Student Task 3: Interpreting the Race
1.  **Model Choice**: Look at the chart. If your goal is to catch as much fraud as possible with only a **10%** false alarm rate (FPR = 0.1), which model performs better?
2.  **The AUC Meaning**: Why is the Random Forest (Model B) 'above' the Logistic Regression? What does this tell you about the model's ability to rank fraud?
3.  **Ideal Spot**: Where on this graph is the 'Point of Perfection'? Does either model get close?

ANSWER TASK 3:
1. O Modelo B, pois ela atinge uma taxa maior de Verdadeiros Positivos quando o limite de falso positivos é de exatos 10%.

2. Estar acima da outra indica uma maior Área Sob a Curva (AUC). Isso prova que a Random Forest é superior em ranquear as probabilidades (separar o que é fraude do que é legítimo e etc).

3.  É o canto superior esquerdo do gráfico, que imite um quadrado. O que mais chega perto é Random Forest.

### Case 4: Averaging & Segmentation (Meta/NIH)

#### 🍱 Multi-class Challenges
In many real-world tasks (like categorizing customer support tickets or identifying cell types), we have more than two classes.
How do we summarize performance across all of them?



In [19]:
# 1. Multi-class dataset (3 classes: Major, Minor, Rare)
Xm, ym = make_classification(n_samples=1000, n_classes=3, weights=[0.7, 0.2, 0.1], n_informative=10, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.3, random_state=42)

clf_multi = RandomForestClassifier(random_state=42).fit(Xm_train, ym_train)
y_pred_m = clf_multi.predict(Xm_test)

# 2. Traditional Multi-class report
print("--- Classification Report ---")
print(metrics.classification_report(ym_test, y_pred_m, target_names=['Major', 'Minor', 'Rare']))

# 3. Viz: Multi-class Confusion Matrix
cm_multi = metrics.confusion_matrix(ym_test, y_pred_m)
plot_unlabeled_cm(ym_test, y_pred_m, "3-Class Confusion Matrix")

--- Classification Report ---
              precision    recall  f1-score   support

       Major       0.80      1.00      0.88       212
       Minor       0.93      0.54      0.68        52
        Rare       1.00      0.14      0.24        36

    accuracy                           0.81       300
   macro avg       0.91      0.56      0.60       300
weighted avg       0.84      0.81      0.77       300



alt.LayerChart(...)

#### 🛠 Student Task 4.1: Interpreting Averages
Look at the **Classification Report** above.
1.  Which class was the hardest for the model to identify? Why?
2.  Compare the **macro avg** and the **weighted avg** for F1-score. Why is the weighted average higher? (Hint: Look at the 'support' column).


ANSWER TASK 4.1: A Classe 2 foi a mais difícil. De 36 exemplos, o modelo acertou apenas 5 , confundindo 30 deles com a Classe 0.

A weighted avg é maior que a macro avg porque leva em conta o número de exemplos por classe. Como a Classe 0 domina o dataset (212 amostras) e tem ótimo desempenho, ela eleva a média ponderada. Já a macro avg dá peso igual às classes, e o mau desempenho na Classe 2 reduz significativamente a média geral.

#### ✂️ Semantic Segmentation (Jaccard)
In medical imaging (NIH) or image editing (Meta), we often perform **Segmentation**—trying to identify the exact pixels that belong to a tumor or a person. The **Jaccard Score** (also called **Intersection over Union / IoU**) is the gold standard for measuring how well two 'masks' overlap.

#### 🛠 Student Task 4.2: The Jaccard Manual Calculation
**Context:** Imagine we are identifying a tumor in an X-ray.
-   `mask_true` is the actual tumor area.
-   `mask_pred` is the area our model highlighted.

**Exercise:**
1.  Look at the two arrays below.
2.  Count the **Intersection** (where both are 1).
3.  Count the **Union** (where at least one is 1).
4.  **Calculate Jaccard** = Intersection / Union.
5.  Check your result using `metrics.jaccard_score` in the code cell.
6.  Report your calculations


In [20]:
# Jaccard (IoU)
mask_true = np.array([0, 1, 1, 0, 1])
mask_pred = np.array([0, 1, 0, 0, 1])
print(f"Jaccard Score: {metrics.jaccard_score(mask_true, mask_pred):.4f}")

Jaccard Score: 0.6667


ANSWER TASK 4.2: A interseção é 2, pois o valor 1 coincide em duas posições (2º e 5º elementos), enquanto a união é 3, já que o valor 1 aparece em pelo menos um dos arrays em três posições (2º, 3º e 5º elementos). Assim, o índice de Jaccard é 2/3 quase = 0,6667, confirmando o valor obtido pela função jaccard_score do sklearn.

## 🔸 Part 2: Regression Metrics


### Case 5: Real Estate Valuation (Zillow/California Housing)
**Metric:** R2, MAE, MedAE, Explained Variance.


#### 🏠 The Zillow Problem
When we predict house prices, we aren't picking a category (0 or 1). We are predicting a **continuous value** (e.g., \$450,000). How do we know if our guess is good?

#### 📏 The Metric Toolbox
1.  **$R^2$ Score (Coefficient of Determination)**:
    -   Think of this as **"The Percentage of Variance Explained."**
    -   A score of **1.0** is perfect. A score of **0.0** means your model is no better than a model that simply guesses the average house price every single time.
2.  **MAE (Mean Absolute Error)**:
    -   The **"Average Error"**. It tells you, on average, how many dollars you are off.
    -   In this dataset, prices are in units of **\$100,000**. So an MAE of `0.5` means we are off by **\$50,000** on average.
3.  **MedAE (Median Absolute Error)**:
    -   The **"Typical Error"**. Because it uses the median, it is **robust to outliers**.
    -   If you have 99 modest houses and 1 mansion worth \$10 million, the mansion will ruin your MAE, but MedAE will stay calm.

In [21]:
housing = fetch_california_housing()
housing_df = pd.DataFrame(housing.data, columns=housing.feature_names)
housing_df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32


The price data is returned as a "Bunch" object (similar to a dictionary). The prices are stored in the .target attribute.

The target variable in this specific dataset is expressed in units of $100,000.

A value of 1.0 in the dataset represents $100,000.
A value of 0.5272 in the dataset represents $0.5272 \times 100,000 = \text{$52,720}$.
This is a common practice in machine learning datasets to keep the numbers small (e.g., between 0 and 5) rather than dealing with millions, which helps models converge faster.

In [22]:
housing.target

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894])

In [23]:
print(housing.DESCR)


.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

#### 📍 Note on the 'Missing' Target
In the `DESCR` list above, you see 8 inputs (MedInc, HouseAge, etc.).
The **Target Value** (what we predict) is listed separately.
- **Name**: `MedHouseVal`
- **Units**: Hundreds of thousands of dollars (**$100,000**).

Let's verify this by looking at the raw numbers in `y` (the target).

In [24]:
print(f"First 5 house prices: {housing.target[:5]}")
print(f"Minimum Price in data: {housing.target.min():.4f} ($15,000)")
print(f"Maximum Price in data: {housing.target.max():.4f} ($500,000)")

First 5 house prices: [4.526 3.585 3.521 3.413 3.422]
Minimum Price in data: 0.1500 ($15,000)
Maximum Price in data: 5.0000 ($500,000)


In [25]:
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Train a Linear Regression Model
reg = LinearRegression().fit(X_train, y_train)
y_pred = reg.predict(X_test)

# 3. Calculate Metrics
r2 = metrics.r2_score(y_test, y_pred)
mae = metrics.mean_absolute_error(y_test, y_pred)
medae = metrics.median_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Median Absolute Error (MedAE): {medae:.4f}")

R2 Score: 0.5958
Mean Absolute Error (MAE): 0.5272
Median Absolute Error (MedAE): 0.4090


#### 🛠 Student Task 5: Interpreting the Dollar Value
1.  **R2 Interpretation**: Look at the $R^2$ value. Roughly what percentage of the price variance is our model explaining?
2.  **The Outlier Check**: Is the **MAE** higher than the **MedAE**? If so, does that mean our model is making massive errors on a few 'luxury' homes, or small errors on all homes?
3.  **Real-world context**: If an average house is \$300,000, and our MAE is `0.5` (\$50,000), would you trust this model to set the final selling price for a home?

ANSWER TASK 5: O modelo explica cerca de 60% (R² = 0,5958) da variação nos preços das casas. A presença de outliers é indicada pelo MAE (0,5272) maior que o MedAE (0,4090), sugerindo erros muito altos em poucos casos — provavelmente imóveis de luxo — que elevam a média, enquanto o erro típico permanece menor. Na prática, não é confiável: um erro de cerca de 50.000 dolares em uma casa de 300.000 representa quase 17%, sendo útil apenas para estimativas iniciais, mas arriscado para definir o preço final.

### Case 6: Energy Demand (GE) & Safety (Waymo)
**Metric:** MSE, MaxError.


#### ⚡️ Context: When Errors Mean Danger
In some industries, an "average" error isn't enough. We need to know the **worst-case scenario** or penalize **large crashes** more than small misses.

1.  **Energy (GE)**: Predicting the Megawatts (MW) needed for the city. If we are off by a little, we waste money. If we are off by a lot, we have a blackout.
2.  **Safety (Waymo)**: Predicting the distance to the car in front. An error of 10cm is fine. An error of 10 meters is a collision.

#### 📐 The 'Penalty' Toolkit
1.  **MSE (Mean Squared Error)**: Unlike MAE, this **squares the errors**.
    - Small error (2) $\rightarrow$ Penalty 4.
    - Large error (10) $\rightarrow$ Penalty 100!
    - *Intuition: It hates big mistakes.*
2.  **Max Error**: Literally the **worst guess** the model made in the entire test set.
    - *Intuition: Critical for safety (can we trust the model never to fail catastrophically?).*
3.  **MSLE (Mean Squared Logarithmic Error)**:
    - Useful when the data has exponential growth or large ranges. It cares about the **ratio** of the error rather than the absolute value.
4.  **MAPE (Mean Absolute Percentage Error)**:
    - The most human-friendly metric. "On average, our demand prediction was off by **3.5%**."


In [26]:
# 1. Generate Synthetic Demand Data
X, y = make_regression(n_samples=500, n_features=5, noise=1.0, random_state=42)
y = np.abs(y) + 10 # Ensure positive values for log metrics

# 2. Simulate two models:
# Model A: Consistent small errors.
# Model B: Perfect most of the time, but 2 catastrophic failures (Outliers).
y_test = y[:100]
y_pred_a = y_test + np.random.normal(0, 5, 100)
y_pred_b = y_test.copy()
y_pred_b[0] += 50 # Catastrophic error 1
y_pred_b[50] -= 50 # Catastrophic error 2

# 3. Calculate Metrics for Comparison
def get_stats(true, pred):
    return {
        "MAE": metrics.mean_absolute_error(true, pred),
        "MSE": metrics.mean_squared_error(true, pred),
        "Max Error": metrics.max_error(true, pred),
        "MAPE": metrics.mean_absolute_percentage_error(true, pred)
    }

print("Model A (Consistent):", get_stats(y_test, y_pred_a))
print("Model B (With Disasters):", get_stats(y_test, y_pred_b))

Model A (Consistent): {'MAE': 3.7182236861873035, 'MSE': 21.2759528384057, 'Max Error': np.float64(13.280050456360215), 'MAPE': 0.06217058730424887}
Model B (With Disasters): {'MAE': 1.0, 'MSE': 50.00000000000001, 'Max Error': np.float64(50.00000000000001), 'MAPE': 0.027576911644846946}


#### 🛠 Student Task 6: Picking the Safer Model
1.  **The Comparison**: Look at Model A and Model B. Which one has a lower **MAE**?
2.  **The Safety Choice**: If you are a safety engineer at Waymo, which model would you deploy? Use **MSE** and **Max Error** to justify your choice.
3.  **Percentage**: What is the **MAPE** for Model A? Does "off by X%" feel easier to explain to a manager than "MSE is Y"?



ANSWER TASK 6:

A Comparação: O Modelo B tem o menor MAE.

A Escolha de Segurança: Eu escolheria o Modelo A. O max error do modelo B é gigantesco (50.0) e o seu MSE é mais do que o dobro do Modelo A.

Porcentagem: O MAPE do Modelo A é de 0.0621. Sim.

### Case 7: Relative Growth & Deviance (Shopify/AXA)
**Metric:** MSLE, MAPE, D2 Score.


#### 📈 Context: Small Shops vs. Giant Stores
Imagine you are at **Shopify**, predicting the monthly sales growth for two stores:
-   **Store A (Small)**: Sells \$100/mo. If your model is off by \$20, that's a **20% error**.
-   **Store B (Giant)**: Sells \$1,000,000/mo. If your model is off by \$20, that's almost **0% error**.

Metrics like MAE and MSE would treat that \$20 miss as the "same error," but for a business, they are completely different.

#### 📐 The 'Ratio' Toolkit
1.  **MSLE (Mean Squared Logarithmic Error)**:
    -   Instead of absolute difference, it looks at the **ratio** between prediction and actual.
    -   It is robust to huge ranges (thousands to millions).
    -   *Intuition: It treats a 10% error on a cheap item and a 10% error on an expensive item as equally bad.*
2.  **MAPE (Mean Absolute Percentage Error)**:
    -   Average percent error. Easy to explain to anyone. "Our prediction for the insurance claim (AXA) was off by **4.5%** on average."
3.  **$D^2$ Score (Deviance)**:
    -   A more robust version of $R^2$. It handles datasets with "weird" distributions (like insurance claims or store hits) where errors aren't normally distributed.


In [27]:
# 1. Create Synthetic Growth Data (Exponentially Distributed)
# Stores ranging from $10 to $50,000 in sales
np.random.seed(42)
y_true = np.exp(np.random.uniform(2, 10, 200)) # Log-normal growth

# 2. Simulate a "Percentage-based" Model
# Model B is off by roughly 10% for everyone
y_pred = y_true * np.random.uniform(0.9, 1.1, 200)

# 3. Calculate Metrics
msle = metrics.mean_squared_log_error(y_true, y_pred)
mape = metrics.mean_absolute_percentage_error(y_true, y_pred)
d2 = metrics.d2_absolute_error_score(y_true, y_pred)

print(f"MSLE: {msle:.6f}")
print(f"MAPE: {mape:.2%} (Percentage Error)")
print(f"D2 Score: {d2:.4f}")

MSLE: 0.003322
MAPE: 5.06% (Percentage Error)
D2 Score: 0.9516


In [29]:
msle = metrics.mean_squared_log_error(y_true, np.maximum(y_pred, 0))
mape = metrics.mean_absolute_percentage_error(y_true, y_pred)
print(f"MSLE: {msle:.4f}")
print(f"MAPE: {mape:.4f} ({mape*100:.2f}%)")

MSLE: 0.0033
MAPE: 0.0506 (5.06%)


#### 🛠 Student Task 7: Manual Ratio Intuition
**Scenario:**
-   **Actual Sales**: \$100
-   **Model Guess**: \$110
-   **Relative Error**: 10%

**Calculation Exercise:**
1.  What is the **Absolute Error**? (\$10)
2.  Now imagine a store with \$1,000,000 in sales and a guess of \$1,000,010. The **Absolute Error** is still \$10.
3.  Which of our metrics (MAE, MSLE, or MAPE) would "flag" the first case as being worse than the second? Why?
4.  **Try it!**: In the cell below, calculate the **MAPE** for these two specific points.


In [33]:
# Compare MAPE for small vs large store with same $10 absolute error
y_t = np.array([100, 1000000])
y_p = np.array([110, 1000010])

print(f"MAPE for Small Store: {metrics.mean_absolute_percentage_error([y_t[0]], [y_p[0]]):.2%}")
print(f"MAPE for Large Store: {metrics.mean_absolute_percentage_error([y_t[1]], [y_p[1]]):.6%}")

y_true_small = [100]
y_pred_small = [110]

y_true_large = [1000000]
y_pred_large = [1000010]

mape_small = metrics.mean_absolute_percentage_error(y_true_small, y_pred_small)
mape_large = metrics.mean_absolute_percentage_error(y_true_large, y_pred_large)

print(f"MAPE for Small Store: {mape_small*100:.2f}%")
print(f"MAPE for Large Store: {mape_large*100:.6f}%")

MAPE for Small Store: 10.00%
MAPE for Large Store: 0.001000%
MAPE for Small Store: 10.00%
MAPE for Large Store: 0.001000%


ANSWER TASK 7:
Qual métrica sinaliza o primeiro caso como pior?
O MAPE e tambem o MSLE. Já o MAE e o MSE tratariam os dois casos como iguais, pois enxergam apenas o erro absoluto de US$ 10.

Por quê?
Porque MAPE e MSLE consideram o erro de forma relativa ao valor real. Errar US$ 10 em US$ 100 é um erro de 10%, enquanto o mesmo erro em US$ 1.000.000 é praticamente irrelevante (0,001%). Por isso, métricas proporcionais são mais adequadas quando há grandes diferenças de escala nos dados.